# classify_infocuria_court_cases_v2

> **v2 checkpoint hardening:** malformed/truncated JSONL checkpoint lines are skipped with a warning instead of aborting the run. Checkpoint appends also ensure a line boundary and flush/sync each completed record. The default checkpoint/output filenames remain `..._v1` intentionally so this notebook can resume the existing classification run without discarding completed jobs.

Validate and reconstruct **Court-to-Court case references** from the high-recall regex candidates produced by `run_court_regex_on_infocuria`.

This notebook deliberately performs a narrower task than `classify_infocuria_citations_roles_v9`:

- no Commission-role classification;
- no direct-appeal classification;
- no attempt to discover unrelated citations throughout an entire judgment;
- only determine whether supplied regex candidates really refer to Court cases and reconstruct the complete Court case number(s) represented by their local evidence.

The model is expected to distinguish genuine Court cases from Commission decisions, Council acts, regulations, directives, OJ references, article/paragraph numbers, dates, and other look-alike identifiers.

## Main input

`output/court_infocuria/excel/court_infocuria_court_regex_candidates.xlsx`

The input is already one row per **source document × regex candidate** and contains aggregated local evidence such as `matched_texts`, `context_agg`, and `paragraph_text_agg`.

## Model

The notebook uses the same local OpenAI-compatible vLLM setup as the earlier classification pipeline, with:

`google/gemma-4-31B-it`

## Important extraction philosophy

The supplied candidates are **hints, not answers**.

A candidate may be:

- a genuine explicit Court case such as `C-22/97` or `T-23/22`;
- a historical prefixless Court case such as `85/76`;
- a fragment of coordinated shorthand such as the `23/77` captured inside `Cases 21 to 23/77`;
- malformed by OCR;
- or not a Court case at all.

Gemma must give an explicit outcome for every supplied candidate and may add case numbers only when they are clearly reconstructed from that candidate's supplied evidence, especially coordination, shared prefixes, shared years, or short numeric ranges.


In [1]:
from __future__ import annotations

from pathlib import Path
import asyncio
import json
import os
import re
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import httpx
from tqdm.auto import tqdm
from IPython.display import display


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "output").exists() and (
            (candidate / "input").exists()
            or (candidate / "data").exists()
        ):
            return candidate
        if candidate.name == "eccjeu":
            return candidate
    return start


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

COURT_OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
EXCEL_DIR = COURT_OUTPUT_DIR / "excel"
JSONL_DIR = COURT_OUTPUT_DIR / "jsonl"

EXCEL_DIR.mkdir(parents=True, exist_ok=True)
JSONL_DIR.mkdir(parents=True, exist_ok=True)


def resolve_input_path(filename: str) -> Path:
    candidates = [
        EXCEL_DIR / filename,
        PROJECT_ROOT / filename,
        NOTEBOOK_DIR / filename,
        Path("/mnt/data") / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    searched = "\n".join(
        f"  - {candidate}"
        for candidate in candidates
    )

    raise FileNotFoundError(
        f"Could not find {filename}. Searched:\n{searched}"
    )


INPUT_PATH = resolve_input_path(
    "court_infocuria_court_regex_candidates.xlsx"
)

# v2 intentionally reuses the v1 checkpoint/output names below.
# The classification schema has not changed, so RUN_MODE="update" can
# resume already completed jobs instead of starting from zero.

CHECKPOINT_JSONL_PATH = Path(
    os.getenv(
        "COURT_CASE_CLASSIFICATION_JSONL",
        str(
            JSONL_DIR
            / "court_infocuria_court_case_classification_v1.jsonl"
        ),
    )
).expanduser().resolve()

OUTPUT_XLSX_PATH = Path(
    os.getenv(
        "COURT_CASE_CLASSIFICATION_XLSX",
        str(
            EXCEL_DIR
            / "court_infocuria_court_case_classification_v1.xlsx"
        ),
    )
).expanduser().resolve()


# ---------------------------------------------------------------------
# Local vLLM configuration
# ---------------------------------------------------------------------

VLLM_BASE_URL = os.getenv(
    "VLLM_BASE_URL",
    "http://localhost:8000/v1",
)

VLLM_API_KEY = os.getenv(
    "VLLM_API_KEY",
    "EMPTY",
)

MODEL = os.getenv(
    "VLLM_MODEL",
    "google/gemma-4-31B-it",
)

# update       = run only jobs without a successful checkpoint
# redo_all     = delete checkpoint and rerun every job
# rebuild_excel= make no LLM calls; rebuild XLSX from checkpoint
RUN_MODE = os.getenv(
    "RUN_MODE",
    "update",
).strip().lower()

VALID_RUN_MODES = {
    "update",
    "redo_all",
    "rebuild_excel",
}

if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(
        f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}"
    )


# ---------------------------------------------------------------------
# Inference controls
# ---------------------------------------------------------------------

CONCURRENT_REQUESTS = int(
    os.getenv(
        "CONCURRENT_REQUESTS",
        "6",
    )
)

REQUEST_TIMEOUT_SECONDS = int(
    os.getenv(
        "REQUEST_TIMEOUT_SECONDS",
        "3600",
    )
)

MAX_RETRIES = int(
    os.getenv(
        "MAX_RETRIES",
        "2",
    )
)

TEMPERATURE = float(
    os.getenv(
        "TEMPERATURE",
        "0.0",
    )
)

MAX_OUTPUT_TOKENS = int(
    os.getenv(
        "MAX_OUTPUT_TOKENS",
        "12000",
    )
)


# ---------------------------------------------------------------------
# Candidate-packet controls
# ---------------------------------------------------------------------

# The regex candidate workbook already contains local evidence. Keep the
# useful local context rather than resending complete Court judgments.
MAX_CONTEXT_CHARS_PER_CANDIDATE = int(
    os.getenv(
        "MAX_CONTEXT_CHARS_PER_CANDIDATE",
        "6500",
    )
)

MAX_PARAGRAPH_CHARS_PER_CANDIDATE = int(
    os.getenv(
        "MAX_PARAGRAPH_CHARS_PER_CANDIDATE",
        "2500",
    )
)

# Large documents are split into candidate chunks. Each candidate stays intact.
MAX_CANDIDATES_PER_JOB = int(
    os.getenv(
        "MAX_CANDIDATES_PER_JOB",
        "60",
    )
)

MAX_EVIDENCE_CHARS_PER_JOB = int(
    os.getenv(
        "MAX_EVIDENCE_CHARS_PER_JOB",
        "120000",
    )
)

# Optional quick testing.
MAX_DOCUMENTS = None
# Example:
# MAX_DOCUMENTS = 20


print("PROJECT_ROOT:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Checkpoint:", CHECKPOINT_JSONL_PATH)
print("Excel output:", OUTPUT_XLSX_PATH)
print("Model:", MODEL)
print("RUN_MODE:", RUN_MODE)
print("Concurrent requests:", CONCURRENT_REQUESTS)


PROJECT_ROOT: /home/edik/projects/eccjeu
Input: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_court_regex_candidates.xlsx
Checkpoint: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/court_infocuria_court_case_classification_v1.jsonl
Excel output: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_court_case_classification_v1.xlsx
Model: google/gemma-4-31B-it
RUN_MODE: update
Concurrent requests: 6


## 1. Load the regex candidates

The regex workbook is treated as the high-recall candidate universe. The LLM does not need to rediscover unrelated Court references from the whole judgment.

For each document, the candidate evidence packet preserves:

- source Court case number;
- normalized regex candidate;
- regex family;
- number of regex hits;
- matched surface forms;
- context trigger;
- local aggregated context;
- paragraph evidence where useful.


In [2]:
# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def clean_scalar(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return ""

    return text


def clean_document_id(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    if isinstance(value, int):
        return str(value)

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    text = str(value).strip()

    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]

    return text


def safe_int(
    value: Any,
    default: int = 0,
) -> int:
    try:
        return max(
            int(float(value)),
            0,
        )
    except Exception:
        return default


def clip_text(
    value: Any,
    max_chars: int,
) -> str:
    text = clean_scalar(value)

    if len(text) <= max_chars:
        return text

    half = max_chars // 2

    return (
        text[:half]
        + "\n...[EVIDENCE TRUNCATED]...\n"
        + text[-half:]
    )


def first_nonempty(
    series: pd.Series,
) -> str:
    for value in series:
        text = clean_scalar(value)

        if text:
            return text

    return ""


# ---------------------------------------------------------------------
# Load candidates
# ---------------------------------------------------------------------

candidates = pd.read_excel(
    INPUT_PATH,
)

candidates.columns = [
    str(column).strip()
    for column in candidates.columns
]

required_columns = {
    "document_id",
    "normalized_candidate",
    "regex_families",
    "context_agg",
}

missing_columns = sorted(
    required_columns - set(candidates.columns)
)

if missing_columns:
    raise ValueError(
        "Regex candidate workbook is missing required columns: "
        f"{missing_columns}"
    )

candidates["document_id"] = (
    candidates["document_id"]
    .map(clean_document_id)
)

candidates["normalized_candidate"] = (
    candidates["normalized_candidate"]
    .map(clean_scalar)
)

for column in [
    "source_case_number",
    "court_case",
    "regex_families",
    "hit_types",
    "matched_texts",
    "context_triggers",
    "context_agg",
    "paragraph_text_agg",
    "confidence_precheck",
    "affId_source",
    "publishedId_source",
    "procedureId_source",
]:
    if column not in candidates.columns:
        candidates[column] = ""

    candidates[column] = (
        candidates[column]
        .map(clean_scalar)
    )

if "n_hits" not in candidates.columns:
    candidates["n_hits"] = 1

candidates["n_hits"] = (
    candidates["n_hits"]
    .map(
        lambda value: safe_int(
            value,
            1,
        )
    )
)

if "first_hit_start" not in candidates.columns:
    candidates["first_hit_start"] = pd.NA

if "is_exact_self_reference" not in candidates.columns:
    candidates["is_exact_self_reference"] = False


# Remove rows that cannot possibly be useful.
candidates = candidates[
    candidates["document_id"].ne("")
    & candidates["normalized_candidate"].ne("")
].copy()


# Defensive de-duplication.
if "document_case_key" in candidates.columns:
    candidates = (
        candidates
        .drop_duplicates(
            subset=["document_case_key"],
            keep="first",
        )
        .copy()
    )
else:
    candidates = (
        candidates
        .drop_duplicates(
            subset=[
                "document_id",
                "normalized_candidate",
            ],
            keep="first",
        )
        .copy()
    )


if MAX_DOCUMENTS is not None:
    selected_documents = (
        candidates["document_id"]
        .drop_duplicates()
        .head(
            int(MAX_DOCUMENTS)
        )
    )

    candidates = candidates[
        candidates["document_id"].isin(
            set(selected_documents)
        )
    ].copy()


print(
    "Regex candidate rows:",
    f"{len(candidates):,}",
)

print(
    "Source documents:",
    f"{candidates['document_id'].nunique():,}",
)

print(
    "Distinct candidate strings:",
    f"{candidates['normalized_candidate'].nunique():,}",
)

display(
    candidates[
        [
            "document_id",
            "source_case_number",
            "normalized_candidate",
            "regex_families",
            "n_hits",
            "context_triggers",
            "is_exact_self_reference",
        ]
    ].head(20)
)


Regex candidate rows: 134,317
Source documents: 6,321
Distinct candidate strings: 12,733


,document_id,source_case_number,normalized_candidate,regex_families,n_hits,context_triggers,is_exact_self_reference
0,1027522,C-105/02,C-105/02,explicit_ct,1,,True
1,1027522,C-105/02,C-10/00,explicit_ct,1,,False
2,1027522,C-105/02,C-104/02,explicit_ct,1,,False
3,1027522,C-105/02,C-256/98,explicit_ct,1,,False
4,1027522,C-105/02,C-508/03,explicit_ct,1,,False
5,1027522,C-105/02,C-365/97,explicit_ct,1,,False
6,1027522,C-105/02,C-441/02,explicit_ct,1,,False
7,1027522,C-105/02,C-392/02,explicit_ct,1,,False
8,1027522,C-105/02,C-78/01,explicit_ct,1,,False
9,1027522,C-105/02,C-499/99,explicit_ct,1,,False


## 2. Court-case interpretation rules

The model is asked to recognize the **structure** of an EU Court case reference, not merely to approve regex-shaped strings.

### In scope

Only cases of:

- **Court of Justice** — normally `C-...`;
- **General Court / former Court of First Instance** — normally `T-...`;
- historical Court cases where the prefix is absent but the surrounding language establishes that the number is a Court case.

`F-` Civil Service Tribunal cases are intentionally out of scope.

### Typical structure

A reference normally contains:

`[C or T prefix] + [case number] + [year]`

Examples:

- `C-22/97`
- `T-23/22`
- `C 22/97`
- `C–22/97`
- historical `85/76`

OCR/document extraction may damage separators, so the model may recover forms such as `C 22-97` **only when the surrounding evidence clearly shows a Court case**.

The number is normally one to three digits but may contain four. The year is normally two digits and may occasionally appear as four digits.

### Positive contextual evidence

Useful evidence includes, among other things:

- `Court`, `Court of Justice`;
- `General Court`;
- `Court of First Instance`;
- `Case`, `Cases`, `Joined Cases`;
- `judgment`, `order`, `appeal`;
- party-name structures such as `X v Commission`;
- wording clearly describing judicial proceedings.

`Court` nearby is strongly helpful, but it is **not mandatory** if other Court-procedure language makes the reference clear.

### Negative contextual evidence

A number must not be accepted merely because it resembles a Court case.

Reject references shown by the evidence to be:

- Commission decisions or Commission administrative case numbers;
- Council, Parliament, High Authority, or other institutional acts;
- regulations or directives;
- Official Journal references;
- Treaty/article/paragraph numbers;
- dates;
- administrative, application, file, or notice numbers;
- `F-` Civil Service Tribunal cases.

### Historical prefixless references

Prefixless references are expected primarily for older cases. In this pipeline, bare historical regex candidates normally have years from `50` through `98`.

Do **not** invent `C-` or `T-` when the source omits the prefix. Preserve a genuine historical citation as, for example, `85/76`.

### Coordination and ranges

The model should reconstruct case numbers that are **clearly implied by the same Court-case expression**.

Examples:

- `Cases 21 to 23/77`
  → `21/77`, `22/77`, `23/77`

- `Cases C-21/77, 22/77 and 23/77`
  → `C-21/77`, `C-22/77`, `C-23/77`

- `Cases 21, 22 and 23/77`
  → `21/77`, `22/77`, `23/77`

- `C-21 to C-23/77`
  → `C-21/77`, `C-22/77`, `C-23/77`

Recognize coordination signals such as commas, `and`, `or`, `to`, dashes, and equivalent multilingual conjunctions.

Only inherit a prefix or year when the grammar clearly coordinates the references.

Only expand **short, plausible integer ranges**. Do not turn a suspicious OCR expression into dozens or hundreds of invented cases.

### Suffixes

Appeal/procedural suffixes such as `P`, `R`, `RENV`, `DEP`, etc. are not needed for this task.

For example:

`C-413/14 P` → canonical case number `C-413/14`.

### Scope of model discovery

The model may reconstruct additional Court case numbers only when they are directly implied by the supplied evidence for a regex candidate.

It should **not** roam through the document looking for unrelated cases that regex did not supply.


In [3]:
# ---------------------------------------------------------------------
# Output vocabularies
# ---------------------------------------------------------------------

CONFIDENCE_VALUES = {
    "high",
    "medium",
    "low",
}

INVALID_REASONS = {
    "none",
    "commission_decision",
    "council_or_parliament_act",
    "regulation_or_directive",
    "official_journal_reference",
    "article_or_paragraph",
    "date_or_year",
    "administrative_or_file_number",
    "civil_service_tribunal_f_case",
    "not_court_case",
    "insufficient_evidence",
    "malformed_unresolvable",
}

REFERENCE_FORMS = {
    "explicit_prefixed",
    "legacy_unprefixed",
    "malformed_recovered",
    "coordination_fragment",
    "none",
}

REASON_ADDED_VALUES = {
    "supplied",
    "expanded_from_coordination",
    "recovered_from_malformed_candidate",
}


# ---------------------------------------------------------------------
# Human-readable target shape
# ---------------------------------------------------------------------

TARGET_SCHEMA = {
    "document_id": "1027522",
    "source_case_number": "C-105/02",
    "chunk_index": 1,
    "chunk_count": 1,
    "document_confidence": "high",
    "candidate_assessments": [
        {
            "provided_candidate": "C-10/00",
            "candidate_is_valid": True,
            "invalid_reason": "none",
            "reference_form": "explicit_prefixed",
            "confidence": "high",
            "best_evidence": "judgment ... Case C-10/00 ...",
            "explanation": "The context explicitly identifies a Court case.",
            "court_cases": [
                {
                    "case_number": "C-10/00",
                    "surface_text": "C-10/00",
                    "prefix": "C",
                    "case_number_component": "10",
                    "year_component": "00",
                    "reason_added": "supplied",
                    "is_inferred_from_coordination": False,
                    "inherited_elements": [],
                    "is_self_reference": False,
                    "confidence": "high",
                    "best_evidence": "judgment ... Case C-10/00 ...",
                    "explanation": "Explicit Court of Justice case reference.",
                }
            ],
        }
    ],
}


# ---------------------------------------------------------------------
# vLLM JSON Schema
# ---------------------------------------------------------------------

COURT_CASE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "case_number": {
            "type": "string",
        },
        "surface_text": {
            "type": "string",
        },
        "prefix": {
            "type": "string",
            "enum": [
                "C",
                "T",
                "",
            ],
        },
        "case_number_component": {
            "type": "string",
        },
        "year_component": {
            "type": "string",
        },
        "reason_added": {
            "type": "string",
            "enum": sorted(
                REASON_ADDED_VALUES
            ),
        },
        "is_inferred_from_coordination": {
            "type": "boolean",
        },
        "inherited_elements": {
            "type": "array",
            "items": {
                "type": "string",
            },
        },
        "is_self_reference": {
            "type": "boolean",
        },
        "confidence": {
            "type": "string",
            "enum": sorted(
                CONFIDENCE_VALUES
            ),
        },
        "best_evidence": {
            "type": "string",
        },
        "explanation": {
            "type": "string",
        },
    },
    "required": [
        "case_number",
        "surface_text",
        "prefix",
        "case_number_component",
        "year_component",
        "reason_added",
        "is_inferred_from_coordination",
        "inherited_elements",
        "is_self_reference",
        "confidence",
        "best_evidence",
        "explanation",
    ],
    "additionalProperties": False,
}


CANDIDATE_ASSESSMENT_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "provided_candidate": {
            "type": "string",
        },
        "candidate_is_valid": {
            "type": "boolean",
        },
        "invalid_reason": {
            "type": "string",
            "enum": sorted(
                INVALID_REASONS
            ),
        },
        "reference_form": {
            "type": "string",
            "enum": sorted(
                REFERENCE_FORMS
            ),
        },
        "confidence": {
            "type": "string",
            "enum": sorted(
                CONFIDENCE_VALUES
            ),
        },
        "best_evidence": {
            "type": "string",
        },
        "explanation": {
            "type": "string",
        },
        "court_cases": {
            "type": "array",
            "items": (
                COURT_CASE_JSON_SCHEMA
            ),
        },
    },
    "required": [
        "provided_candidate",
        "candidate_is_valid",
        "invalid_reason",
        "reference_form",
        "confidence",
        "best_evidence",
        "explanation",
        "court_cases",
    ],
    "additionalProperties": False,
}


RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "document_id": {
            "type": "string",
        },
        "source_case_number": {
            "type": "string",
        },
        "chunk_index": {
            "type": "integer",
            "minimum": 1,
        },
        "chunk_count": {
            "type": "integer",
            "minimum": 1,
        },
        "document_confidence": {
            "type": "string",
            "enum": sorted(
                CONFIDENCE_VALUES
            ),
        },
        "candidate_assessments": {
            "type": "array",
            "items": (
                CANDIDATE_ASSESSMENT_JSON_SCHEMA
            ),
        },
    },
    "required": [
        "document_id",
        "source_case_number",
        "chunk_index",
        "chunk_count",
        "document_confidence",
        "candidate_assessments",
    ],
    "additionalProperties": False,
}


TARGET_SCHEMA


{'document_id': '1027522',
 'source_case_number': 'C-105/02',
 'chunk_index': 1,
 'chunk_count': 1,
 'document_confidence': 'high',
 'candidate_assessments': [{'provided_candidate': 'C-10/00',
   'candidate_is_valid': True,
   'invalid_reason': 'none',
   'reference_form': 'explicit_prefixed',
   'confidence': 'high',
   'best_evidence': 'judgment ... Case C-10/00 ...',
   'explanation': 'The context explicitly identifies a Court case.',
   'court_cases': [{'case_number': 'C-10/00',
     'surface_text': 'C-10/00',
     'prefix': 'C',
     'case_number_component': '10',
     'year_component': '00',
     'reason_added': 'supplied',
     'is_inferred_from_coordination': False,
     'inherited_elements': [],
     'is_self_reference': False,
     'confidence': 'high',
     'best_evidence': 'judgment ... Case C-10/00 ...',
     'explanation': 'Explicit Court of Justice case reference.'}]}]}

## 3. Build document-level candidate evidence packets

The default is **one LLM job per source document**.

If a document contains too many candidates or too much aggregated evidence, it is split into candidate chunks. A candidate is never split across jobs.

This preserves the main advantage of document-level reasoning: Gemma can see related candidates together and understand coordinated shorthand.


In [4]:
# ---------------------------------------------------------------------
# Candidate evidence packet
# ---------------------------------------------------------------------

def source_case_from_group(
    group: pd.DataFrame,
) -> str:
    for column in [
        "source_case_number",
        "court_case",
    ]:
        value = first_nonempty(
            group[column]
        )

        if value:
            return value

    return ""


def build_candidate_record(
    row: pd.Series,
) -> dict[str, Any]:

    context = clip_text(
        row.get(
            "context_agg",
            "",
        ),
        MAX_CONTEXT_CHARS_PER_CANDIDATE,
    )

    paragraph = clip_text(
        row.get(
            "paragraph_text_agg",
            "",
        ),
        MAX_PARAGRAPH_CHARS_PER_CANDIDATE,
    )

    # The ±regex context is normally the most useful evidence.
    # Paragraph text is supplementary and omitted if identical.
    if paragraph == context:
        paragraph = ""

    return {
        "provided_candidate": clean_scalar(
            row.get(
                "normalized_candidate"
            )
        ),
        "regex_family": clean_scalar(
            row.get(
                "regex_families"
            )
        ),
        "hit_types": clean_scalar(
            row.get(
                "hit_types"
            )
        ),
        "n_regex_hits": safe_int(
            row.get(
                "n_hits"
            ),
            1,
        ),
        "matched_texts": clean_scalar(
            row.get(
                "matched_texts"
            )
        ),
        "context_trigger": clean_scalar(
            row.get(
                "context_triggers"
            )
        ),
        "regex_confidence_precheck": clean_scalar(
            row.get(
                "confidence_precheck"
            )
        ),
        "regex_self_reference_precheck": bool(
            row.get(
                "is_exact_self_reference",
                False,
            )
        ),
        "first_hit_start": (
            None
            if pd.isna(
                row.get(
                    "first_hit_start"
                )
            )
            else safe_int(
                row.get(
                    "first_hit_start"
                )
            )
        ),
        "context": context,
        "paragraph_evidence": paragraph,
    }


def candidate_record_chars(
    record: dict[str, Any],
) -> int:
    return len(
        json.dumps(
            record,
            ensure_ascii=False,
        )
    )


def split_candidate_records(
    records: list[dict[str, Any]],
) -> list[list[dict[str, Any]]]:

    chunks: list[
        list[dict[str, Any]]
    ] = []

    current: list[
        dict[str, Any]
    ] = []

    current_chars = 0

    for record in records:
        record_chars = (
            candidate_record_chars(
                record
            )
        )

        would_exceed_count = (
            len(current)
            >= MAX_CANDIDATES_PER_JOB
        )

        would_exceed_chars = (
            bool(current)
            and (
                current_chars
                + record_chars
                > MAX_EVIDENCE_CHARS_PER_JOB
            )
        )

        if (
            would_exceed_count
            or would_exceed_chars
        ):
            chunks.append(current)
            current = []
            current_chars = 0

        current.append(record)
        current_chars += record_chars

    if current:
        chunks.append(current)

    return chunks


document_jobs: list[
    dict[str, Any]
] = []

for document_id, group in tqdm(
    candidates.groupby(
        "document_id",
        sort=False,
    ),
    desc="Building document jobs",
):

    group = group.sort_values(
        "first_hit_start",
        na_position="last",
    )

    source_case_number = (
        source_case_from_group(
            group
        )
    )

    candidate_records = [
        build_candidate_record(row)
        for _, row
        in group.iterrows()
    ]

    chunks = split_candidate_records(
        candidate_records
    )

    chunk_count = len(chunks)

    metadata = {
        "source_case_number": (
            source_case_number
        ),
        "affId_source": first_nonempty(
            group["affId_source"]
        ),
        "publishedId_source": first_nonempty(
            group[
                "publishedId_source"
            ]
        ),
        "procedureId_source": first_nonempty(
            group[
                "procedureId_source"
            ]
        ),
        "n_regex_candidates_document": int(
            len(group)
        ),
    }

    for chunk_index, chunk in enumerate(
        chunks,
        start=1,
    ):
        job_id = (
            document_id
            if chunk_count == 1
            else (
                f"{document_id}"
                f"::chunk_{chunk_index:03d}"
            )
        )

        document_jobs.append({
            "job_id": job_id,
            "document_id": document_id,
            "source_case_number": (
                source_case_number
            ),
            "chunk_index": (
                chunk_index
            ),
            "chunk_count": (
                chunk_count
            ),
            "input_strategy": (
                "document_candidates"
                if chunk_count == 1
                else "candidate_chunk"
            ),
            "metadata": metadata,
            "supplied_candidates": chunk,
            "n_supplied_candidates": len(
                chunk
            ),
            "evidence_chars": sum(
                candidate_record_chars(
                    record
                )
                for record in chunk
            ),
        })


job_df = pd.DataFrame(
    [
        {
            "job_id": job["job_id"],
            "document_id": (
                job["document_id"]
            ),
            "source_case_number": (
                job[
                    "source_case_number"
                ]
            ),
            "chunk_index": (
                job["chunk_index"]
            ),
            "chunk_count": (
                job["chunk_count"]
            ),
            "input_strategy": (
                job["input_strategy"]
            ),
            "n_supplied_candidates": (
                job[
                    "n_supplied_candidates"
                ]
            ),
            "evidence_chars": (
                job["evidence_chars"]
            ),
        }
        for job in document_jobs
    ]
)


print(
    "Source documents:",
    f"{candidates['document_id'].nunique():,}",
)

print(
    "LLM jobs:",
    f"{len(document_jobs):,}",
)

print(
    "Documents split into >1 job:",
    f"{job_df.groupby('document_id')['job_id'].count().gt(1).sum():,}",
)

print(
    "Largest candidate count in one job:",
    (
        int(
            job_df[
                "n_supplied_candidates"
            ].max()
        )
        if len(job_df)
        else 0
    ),
)

print(
    "Largest evidence packet:",
    (
        f"{int(job_df['evidence_chars'].max()):,} chars"
        if len(job_df)
        else "0"
    ),
)

display(
    job_df.sort_values(
        "evidence_chars",
        ascending=False,
    ).head(20)
)


Building document jobs:   0%|          | 0/6321 [00:00<?, ?it/s]

Source documents: 6,321
LLM jobs: 8,378
Documents split into >1 job: 1,237
Largest candidate count in one job: 60
Largest evidence packet: 119,999 chars


,job_id,document_id,source_case_number,chunk_index,chunk_count,input_strategy,n_supplied_candidates,evidence_chars
6342,619015::chunk_002,619015,T-245/01,2,4,candidate_chunk,35,119999
6330,619012::chunk_002,619012,T-236/01,2,4,candidate_chunk,35,119999
6354,619018::chunk_002,619018,T-252/01,2,4,candidate_chunk,35,119999
6334,619013::chunk_002,619013,T-239/01,2,4,candidate_chunk,35,119999
6338,619014::chunk_002,619014,T-244/01,2,4,candidate_chunk,35,119999
6350,619017::chunk_002,619017,T-251/01,2,4,candidate_chunk,35,119999
6346,619016::chunk_002,619016,T-246/01,2,4,candidate_chunk,35,119999
8309,724745::chunk_001,724745,T-16/96,1,2,candidate_chunk,35,119996
7596,642981::chunk_001,642981,T-57/01,1,4,candidate_chunk,34,119996
2885,2195525::chunk_001,2195525,T-758/14,1,2,candidate_chunk,32,119996


## 4. Prompt

A crucial distinction from the earlier Commission-classification notebook is that Gemma is **not** asked to search the entire judgment.

It receives the complete set (or a chunk) of regex candidates for the document plus the local evidence already captured by the regex pass.

Every supplied candidate must receive exactly one assessment.


In [5]:
SYSTEM_MESSAGE = """
You are a careful legal-data extraction system for EU Court case citations.
Follow the requested JSON schema exactly.
Base every conclusion only on the supplied candidate evidence.
Do not add facts from memory.
Return JSON only.
""".strip()


COURT_CASE_RULES = """
TASK
For every supplied regex candidate, decide whether the supplied evidence shows
a genuine case before the Court of Justice or the General Court / former Court
of First Instance. If it does, reconstruct every complete Court case number
clearly represented by that candidate's local evidence.

The supplied candidates were generated by regex for high recall. They are hints,
not authoritative answers. Some are false positives, fragments, OCR-damaged,
self-references, Commission or Council decisions, regulations, directives,
Official Journal references, article/paragraph numbers, dates, or other
administrative identifiers.

IN-SCOPE COURTS
- Court of Justice cases: normally prefix C.
- General Court / former Court of First Instance cases: normally prefix T.
- Historical Court cases may be genuinely prefixless.
- F-prefixed Civil Service Tribunal cases are OUT OF SCOPE.

TYPICAL CASE STRUCTURE
A modern Court case usually looks like:
  C-22/97
  T-23/22

The prefix is C or T.
The case number is normally 1-3 digits but may be 4 digits.
The year is normally 2 digits and may occasionally appear as 4 digits.

OCR or text extraction can damage separators. Examples such as
  C 22/97
  C–22/97
  C-22-97
may represent C-22/97 when the surrounding evidence clearly shows a Court case.

Do not accept a malformed number merely because it can be repaired into a
plausible shape. Context must support the Court-case interpretation.

YEAR NORMALIZATION
Return the canonical year as two digits.
For example:
  C-22/1997 -> C-22/97
  T-123/2022 -> T-123/22
when the four-digit year is unambiguous.

SUFFIXES
Ignore procedural suffixes such as P, R, RENV, DEP, P-DEP, etc.
For this task:
  C-413/14 P -> C-413/14

POSITIVE COURT EVIDENCE
Useful evidence includes, but is not limited to:
- Court
- Court of Justice
- General Court
- Court of First Instance
- Case, Cases, Joined Cases
- judgment, order, appeal
- X v Commission or another party-name construction
- clear wording describing judicial proceedings

The word "Court" nearby is strong positive evidence but is not mandatory when
other judicial language makes the interpretation clear.

NEGATIVE / OUT-OF-SCOPE EVIDENCE
Reject a candidate when the supplied evidence identifies it as:
- a Commission decision or Commission administrative case number;
- a Council, Parliament, High Authority, or other institutional act;
- a regulation or directive;
- an Official Journal reference;
- a Treaty article, article, paragraph, recital, or similar provision;
- a date or year;
- an administrative, file, application, notice, or personnel number;
- an F-prefixed Civil Service Tribunal case;
- or otherwise not a Court case.

A number's shape alone is never enough.

HISTORICAL PREFIXLESS CASES
Historical references may appear as:
  Case 6/64
  85/76
  Joined Cases 56/64 and 58/64

The prefix may genuinely be absent. Do NOT invent C or T merely because a
historical case would probably belong to one Court.

In this regex pipeline, prefixless candidates normally have years 50-98.
A prefixless number needs Court-case context such as Case/Cases, Court,
judgment/order wording, party names, or equivalent judicial evidence.

COORDINATION AND SHORTHAND
Reconstruct references clearly implied by the same Court-case expression.

Examples:
  Cases 21 to 23/77
    -> 21/77, 22/77, 23/77

  Cases C-21/77, 22/77 and 23/77
    -> C-21/77, C-22/77, C-23/77

  Cases 21, 22 and 23/77
    -> 21/77, 22/77, 23/77

  C-21 to C-23/77
    -> C-21/77, C-22/77, C-23/77

Recognize commas, and, or, to, hyphens/en-dashes used as ranges, and equivalent
multilingual conjunctions such as et, ou, und, oder, e, o, en, of, ja, vagy,
και, y, and similar coordination markers.

Inherit a prefix or year only when the grammar clearly coordinates the cases.

Expand only SHORT AND PLAUSIBLE integer ranges. If an OCR-damaged expression
would require a large or doubtful expansion, do not invent cases; mark the
candidate low-confidence or invalid as appropriate.

MODEL DISCOVERY LIMIT
Do not search for unrelated Court cases beyond the supplied evidence.
You may add a case not present as its own regex candidate only when it is:
- necessarily reconstructed from a supplied candidate's coordinated expression;
- inherited from a clearly shared prefix/year;
- or recovered from an OCR/malformed form represented in that candidate evidence.

SELF REFERENCES
The source Court case number is supplied separately.
A reference to the source case itself is still a VALID Court case reference.
Set is_self_reference=true for that reconstructed case.
Do not reject self-references.

CANDIDATE OUTCOME
Return exactly ONE candidate_assessment for every supplied provided_candidate.
Do not omit supplied candidates and do not invent extra candidate_assessment keys.

If candidate_is_valid=false:
- court_cases must be [];
- reference_form must be none unless malformed_unresolvable is the reason;
- invalid_reason must explain the exclusion.

If candidate_is_valid=true:
- invalid_reason must be none;
- court_cases must contain at least one reconstructed Court case.

CONFIDENCE
high:
  explicit Court-case wording and a clear case number.
medium:
  coordination, shared-year/prefix inheritance, historical shorthand, or modest
  OCR repair is required.
low:
  genuine ambiguity remains but the best reading is still a Court case.

A case reconstructed from coordination should normally be medium confidence,
unless it is independently explicit in the same evidence.

EVIDENCE
best_evidence should quote only a short decisive excerpt from the supplied
candidate evidence.
Do not use outside facts.
""".strip()


def json_safe(
    value: Any,
) -> Any:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    return str(value)


def build_prompt(
    job: dict[str, Any],
) -> str:

    supplied_json = json.dumps(
        job[
            "supplied_candidates"
        ],
        ensure_ascii=False,
        indent=2,
    )

    metadata_json = json.dumps(
        {
            key: json_safe(value)
            for key, value
            in job[
                "metadata"
            ].items()
        },
        ensure_ascii=False,
        indent=2,
    )

    schema_json = json.dumps(
        TARGET_SCHEMA,
        ensure_ascii=False,
        indent=2,
    )

    chunk_notice = (
        "This request contains every regex candidate for the source document."
        if job["chunk_count"] == 1
        else (
            "This is one candidate chunk from a document with many regex "
            "candidates. Classify only the supplied candidates in this chunk."
        )
    )

    return f"""
You are analysing regex-generated candidate references from one EU Court document.

{chunk_notice}

{COURT_CASE_RULES}

EXACT OUTPUT SHAPE
{schema_json}

DOCUMENT ID
{job["document_id"]}

SOURCE COURT CASE NUMBER
{job["source_case_number"]}

CHUNK
{job["chunk_index"]} of {job["chunk_count"]}

DOCUMENT METADATA
{metadata_json}

SUPPLIED REGEX CANDIDATES AND LOCAL EVIDENCE
{supplied_json}

Return JSON only.
""".strip()


# Preview a real prompt without contacting vLLM.
if document_jobs:
    preview_prompt = build_prompt(
        document_jobs[0]
    )

    print(
        preview_prompt[:12000]
    )


You are analysing regex-generated candidate references from one EU Court document.

This request contains every regex candidate for the source document.

TASK
For every supplied regex candidate, decide whether the supplied evidence shows
a genuine case before the Court of Justice or the General Court / former Court
of First Instance. If it does, reconstruct every complete Court case number
clearly represented by that candidate's local evidence.

The supplied candidates were generated by regex for high recall. They are hints,
not authoritative answers. Some are false positives, fragments, OCR-damaged,
self-references, Commission or Council decisions, regulations, directives,
Official Journal references, article/paragraph numbers, dates, or other
administrative identifiers.

IN-SCOPE COURTS
- Court of Justice cases: normally prefix C.
- General Court / former Court of First Instance cases: normally prefix T.
- Historical Court cases may be genuinely prefixless.
- F-prefixed Civil Service

## 5. Deterministic normalization and response validation

Gemma performs the contextual reasoning. Python then enforces simple structural rules:

- only `C`, `T`, or genuinely prefixless forms survive;
- `F` cases are excluded;
- dash variants are normalized;
- four-digit years are reduced to two digits;
- suffixes after the year are ignored;
- self-reference is recalculated deterministically from the source case number;
- every supplied candidate must receive exactly one model assessment.

This deliberately keeps the semantic decision with the model while making the final identifiers consistent.


In [6]:
try:
    from json_repair import (
        repair_json as _repair_json,
    )
except ImportError:
    _repair_json = None


CASE_HYPHENS = r"\-‐‑‒–—−"


def extract_json_object(
    raw_text: str,
) -> dict[str, Any]:

    text = str(
        raw_text
        or ""
    ).strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        parsed = json.loads(text)

        if isinstance(
            parsed,
            dict,
        ):
            return parsed

    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if (
        start < 0
        or end <= start
    ):
        raise ValueError(
            "No JSON object found "
            "in model response"
        )

    candidate = text[
        start:end + 1
    ]

    try:
        parsed = json.loads(
            candidate
        )

    except json.JSONDecodeError:
        if _repair_json is None:
            raise

        repaired = _repair_json(
            candidate
        )

        parsed = (
            json.loads(repaired)
            if isinstance(
                repaired,
                str,
            )
            else repaired
        )

    if not isinstance(
        parsed,
        dict,
    ):
        raise ValueError(
            "Parsed JSON response "
            "is not an object"
        )

    return parsed


def normalize_string_list(
    value: Any,
) -> list[str]:

    if value is None:
        return []

    values = (
        value
        if isinstance(
            value,
            list,
        )
        else [value]
    )

    output = []

    for item in values:
        text = clean_scalar(
            item
        )

        if (
            text
            and text not in output
        ):
            output.append(text)

    return output


def canonical_year(
    value: str,
) -> str | None:

    text = re.sub(
        r"\D",
        "",
        str(
            value
            or ""
        ),
    )

    if len(text) == 2:
        return text

    if (
        len(text) == 4
        and 1950
        <= int(text)
        <= 2099
    ):
        return text[-2:]

    return None


def normalize_court_case(
    value: Any,
) -> str | None:
    """
    Conservative canonicalizer for model-produced case identifiers.

    Valid outputs:
      C-22/97
      T-23/22
      85/76

    Procedural suffixes after the year are ignored.
    """
    text = clean_scalar(
        value
    )

    if not text:
        return None

    text = (
        text
        .upper()
        .replace("CASE", " ")
        .strip()
    )

    # Reject F cases explicitly.
    if re.search(
        rf"(?<![A-Z0-9])F\s*[{CASE_HYPHENS}]?\s*\d",
        text,
    ):
        return None

    # Prefix is optional. Accept common separator/OCR damage.
    match = re.search(
        rf"""
        (?<![A-Z0-9])
        (?:(?P<prefix>[CT])\s*[{CASE_HYPHENS}]?\s*)?
        (?P<number>\d{{1,4}})
        \s*
        (?:/|[{CASE_HYPHENS}])
        \s*
        (?P<year>\d{{2}}|\d{{4}})
        (?!\d)
        """,
        text,
        flags=re.VERBOSE,
    )

    if not match:
        return None

    prefix = (
        match.group(
            "prefix"
        )
        or ""
    )

    number = str(
        int(
            match.group(
                "number"
            )
        )
    )

    year = canonical_year(
        match.group(
            "year"
        )
    )

    if year is None:
        return None

    if prefix:
        return (
            f"{prefix}-{number}/{year}"
        )

    # This pipeline intentionally treats prefixless references as historical.
    if not (
        50
        <= int(year)
        <= 98
    ):
        return None

    return f"{number}/{year}"


def normalized_case_parts(
    case_number: str,
) -> tuple[
    str,
    str,
    str,
]:

    canonical = normalize_court_case(
        case_number
    )

    if canonical is None:
        return "", "", ""

    match = re.fullmatch(
        r"(?:(?P<prefix>[CT])-)?"
        r"(?P<number>\d{1,4})/"
        r"(?P<year>\d{2})",
        canonical,
    )

    if not match:
        return "", "", ""

    return (
        match.group(
            "prefix"
        )
        or "",
        match.group(
            "number"
        ),
        match.group(
            "year"
        ),
    )


def normalize_source_case(
    source_case: Any,
) -> str | None:
    # Same canonicalizer naturally ignores suffixes such as P/R.
    return normalize_court_case(
        source_case
    )


def parse_bool(
    value: Any,
) -> bool:

    if isinstance(
        value,
        bool,
    ):
        return value

    text = str(
        value
        or ""
    ).strip().lower()

    return text in {
        "1",
        "true",
        "yes",
        "y",
    }


def validate_model_response(
    parsed: dict[str, Any],
    job: dict[str, Any],
) -> dict[str, Any]:

    expected_document_id = (
        job["document_id"]
    )

    parsed[
        "document_id"
    ] = clean_document_id(
        parsed.get(
            "document_id"
        )
        or expected_document_id
    )

    if (
        parsed[
            "document_id"
        ]
        != expected_document_id
    ):
        parsed[
            "returned_document_id"
        ] = parsed[
            "document_id"
        ]

        parsed[
            "document_id"
        ] = expected_document_id

    parsed[
        "source_case_number"
    ] = (
        job[
            "source_case_number"
        ]
    )

    parsed[
        "chunk_index"
    ] = int(
        job[
            "chunk_index"
        ]
    )

    parsed[
        "chunk_count"
    ] = int(
        job[
            "chunk_count"
        ]
    )

    document_confidence = (
        clean_scalar(
            parsed.get(
                "document_confidence"
            )
        ).lower()
    )

    parsed[
        "document_confidence"
    ] = (
        document_confidence
        if document_confidence
        in CONFIDENCE_VALUES
        else "low"
    )

    assessments = parsed.get(
        "candidate_assessments"
    )

    if not isinstance(
        assessments,
        list,
    ):
        raise ValueError(
            "'candidate_assessments' "
            "must be a list"
        )

    expected_candidates = [
        item[
            "provided_candidate"
        ]
        for item
        in job[
            "supplied_candidates"
        ]
    ]

    returned_candidates = [
        clean_scalar(
            assessment.get(
                "provided_candidate"
            )
        )
        for assessment
        in assessments
        if isinstance(
            assessment,
            dict,
        )
    ]

    if (
        len(returned_candidates)
        != len(
            expected_candidates
        )
    ):
        raise ValueError(
            "Model did not return exactly "
            "one assessment per supplied candidate. "
            f"Expected {len(expected_candidates)}, "
            f"received {len(returned_candidates)}."
        )

    if sorted(
        returned_candidates
    ) != sorted(
        expected_candidates
    ):
        raise ValueError(
            "Returned provided_candidate values "
            "do not exactly match supplied candidates."
        )

    source_case = (
        normalize_source_case(
            job[
                "source_case_number"
            ]
        )
    )

    normalized_assessments = []

    for assessment in assessments:
        if not isinstance(
            assessment,
            dict,
        ):
            raise ValueError(
                "candidate_assessment "
                "is not an object"
            )

        provided_candidate = (
            clean_scalar(
                assessment.get(
                    "provided_candidate"
                )
            )
        )

        candidate_is_valid = parse_bool(
            assessment.get(
                "candidate_is_valid"
            )
        )

        invalid_reason = (
            clean_scalar(
                assessment.get(
                    "invalid_reason"
                )
            ).lower()
            or "insufficient_evidence"
        )

        if (
            invalid_reason
            not in INVALID_REASONS
        ):
            invalid_reason = (
                "insufficient_evidence"
            )

        reference_form = (
            clean_scalar(
                assessment.get(
                    "reference_form"
                )
            ).lower()
            or "none"
        )

        if (
            reference_form
            not in REFERENCE_FORMS
        ):
            reference_form = "none"

        confidence = (
            clean_scalar(
                assessment.get(
                    "confidence"
                )
            ).lower()
            or "low"
        )

        if (
            confidence
            not in CONFIDENCE_VALUES
        ):
            confidence = "low"

        model_cases = assessment.get(
            "court_cases"
        )

        if not isinstance(
            model_cases,
            list,
        ):
            model_cases = []

        normalized_cases = []
        seen_cases = set()

        if candidate_is_valid:
            for case in model_cases:
                if not isinstance(
                    case,
                    dict,
                ):
                    continue

                canonical = (
                    normalize_court_case(
                        case.get(
                            "case_number"
                        )
                    )
                )

                if (
                    canonical is None
                    or canonical
                    in seen_cases
                ):
                    continue

                seen_cases.add(
                    canonical
                )

                prefix, number, year = (
                    normalized_case_parts(
                        canonical
                    )
                )

                reason_added = (
                    clean_scalar(
                        case.get(
                            "reason_added"
                        )
                    ).lower()
                )

                if (
                    reason_added
                    not in REASON_ADDED_VALUES
                ):
                    reason_added = (
                        "expanded_from_coordination"
                        if parse_bool(
                            case.get(
                                "is_inferred_from_coordination"
                            )
                        )
                        else "supplied"
                    )

                is_inferred = parse_bool(
                    case.get(
                        "is_inferred_from_coordination"
                    )
                )

                case_confidence = (
                    clean_scalar(
                        case.get(
                            "confidence"
                        )
                    ).lower()
                    or confidence
                )

                if (
                    case_confidence
                    not in CONFIDENCE_VALUES
                ):
                    case_confidence = (
                        confidence
                    )

                # Conservative policy copied from the earlier classifier:
                # inferred coordination should not silently become "high".
                if (
                    is_inferred
                    and case_confidence
                    == "high"
                ):
                    case_confidence = (
                        "medium"
                    )

                normalized_cases.append({
                    "case_number": (
                        canonical
                    ),
                    "surface_text": (
                        clean_scalar(
                            case.get(
                                "surface_text"
                            )
                        )
                    ),
                    "prefix": prefix,
                    "case_number_component": (
                        number
                    ),
                    "year_component": year,
                    "reason_added": (
                        reason_added
                    ),
                    "is_inferred_from_coordination": (
                        is_inferred
                    ),
                    "inherited_elements": (
                        normalize_string_list(
                            case.get(
                                "inherited_elements"
                            )
                        )
                    ),
                    "is_self_reference": (
                        bool(
                            source_case
                            and canonical
                            == source_case
                        )
                    ),
                    "confidence": (
                        case_confidence
                    ),
                    "best_evidence": (
                        clean_scalar(
                            case.get(
                                "best_evidence"
                            )
                        )
                    ),
                    "explanation": (
                        clean_scalar(
                            case.get(
                                "explanation"
                            )
                        )
                    ),
                })

        if (
            candidate_is_valid
            and not normalized_cases
        ):
            candidate_is_valid = False
            invalid_reason = (
                "malformed_unresolvable"
            )
            reference_form = "none"

        if not candidate_is_valid:
            normalized_cases = []

            if invalid_reason == "none":
                invalid_reason = (
                    "insufficient_evidence"
                )

            if (
                invalid_reason
                != "malformed_unresolvable"
            ):
                reference_form = "none"

        else:
            invalid_reason = "none"

        normalized_assessments.append({
            "provided_candidate": (
                provided_candidate
            ),
            "candidate_is_valid": (
                candidate_is_valid
            ),
            "invalid_reason": (
                invalid_reason
            ),
            "reference_form": (
                reference_form
            ),
            "confidence": confidence,
            "best_evidence": (
                clean_scalar(
                    assessment.get(
                        "best_evidence"
                    )
                )
            ),
            "explanation": (
                clean_scalar(
                    assessment.get(
                        "explanation"
                    )
                )
            ),
            "court_cases": (
                normalized_cases
            ),
        })

    parsed[
        "candidate_assessments"
    ] = normalized_assessments

    return parsed


## 6. Preflight QA

Before contacting Gemma, verify that:

- every regex candidate appears in exactly one LLM job;
- no candidate was lost during chunking;
- each job has manageable evidence size;
- every candidate has at least some local evidence.

Warnings about missing paragraph text are harmless when `context_agg` is present.


In [7]:
# ---------------------------------------------------------------------
# Candidate coverage audit
# ---------------------------------------------------------------------

input_keys = set(
    zip(
        candidates[
            "document_id"
        ],
        candidates[
            "normalized_candidate"
        ],
    )
)

job_keys = []

preflight_rows = []

for job in document_jobs:
    for candidate in job[
        "supplied_candidates"
    ]:
        key = (
            job[
                "document_id"
            ],
            candidate[
                "provided_candidate"
            ],
        )

        job_keys.append(key)

        preflight_rows.append({
            "job_id": (
                job[
                    "job_id"
                ]
            ),
            "document_id": (
                job[
                    "document_id"
                ]
            ),
            "provided_candidate": (
                candidate[
                    "provided_candidate"
                ]
            ),
            "regex_family": (
                candidate[
                    "regex_family"
                ]
            ),
            "n_regex_hits": (
                candidate[
                    "n_regex_hits"
                ]
            ),
            "context_chars": len(
                candidate[
                    "context"
                ]
            ),
            "paragraph_chars": len(
                candidate[
                    "paragraph_evidence"
                ]
            ),
            "has_any_evidence": bool(
                candidate[
                    "context"
                ]
                or candidate[
                    "paragraph_evidence"
                ]
            ),
        })


preflight_df = pd.DataFrame(
    preflight_rows
)

job_key_set = set(
    job_keys
)


duplicate_job_keys = (
    pd.Series(
        job_keys
    )
    .duplicated()
    .sum()
)


print(
    "Input document/candidate keys:",
    f"{len(input_keys):,}",
)

print(
    "Prepared document/candidate keys:",
    f"{len(job_key_set):,}",
)

print(
    "Duplicate prepared keys:",
    f"{int(duplicate_job_keys):,}",
)

print(
    "Candidates without any evidence:",
    f"{int((~preflight_df['has_any_evidence']).sum()):,}",
)


assert (
    input_keys
    == job_key_set
), (
    "Candidate coverage changed during job preparation."
)

assert (
    duplicate_job_keys
    == 0
), (
    "A candidate was assigned to more than one job."
)


display(
    preflight_df[
        ~preflight_df[
            "has_any_evidence"
        ]
    ].head(30)
)


Input document/candidate keys: 134,317
Prepared document/candidate keys: 134,317
Duplicate prepared keys: 0
Candidates without any evidence: 0


,job_id,document_id,provided_candidate,regex_family,n_regex_hits,context_chars,paragraph_chars,has_any_evidence


## 7. vLLM structured-output inference

The checkpoint JSONL is the durable result store.

`RUN_MODE="update"` resumes safely by skipping jobs whose latest checkpoint status is `ok`.

`RUN_MODE="redo_all"` deletes the existing checkpoint and reruns everything.

`RUN_MODE="rebuild_excel"` makes no model calls.


In [8]:
def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).isoformat()


def load_latest_checkpoint(
    path: Path,
) -> dict[
    str,
    dict[str, Any],
]:

    latest = {}
    invalid_lines = []

    if not path.exists():
        return latest

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(
                    line
                )

            except json.JSONDecodeError as exc:
                invalid_lines.append({
                    "line_number": line_number,
                    "error": str(exc),
                    "preview": line[:200],
                })
                continue

            if not isinstance(
                record,
                dict,
            ):
                invalid_lines.append({
                    "line_number": line_number,
                    "error": (
                        "Checkpoint record is not a JSON object"
                    ),
                    "preview": line[:200],
                })
                continue

            job_id = clean_scalar(
                record.get(
                    "job_id"
                )
            )

            if not job_id:
                continue

            previous = latest.get(
                job_id
            )

            if (
                previous is None
                or clean_scalar(
                    record.get(
                        "classified_at"
                    )
                )
                >= clean_scalar(
                    previous.get(
                        "classified_at"
                    )
                )
            ):
                latest[
                    job_id
                ] = record

    if invalid_lines:
        print(
            "WARNING: skipped",
            len(invalid_lines),
            "malformed checkpoint line(s) in",
            path.name,
        )

        for item in invalid_lines[:10]:
            print(
                "  line",
                item["line_number"],
                "-",
                item["error"],
                "- preview:",
                repr(
                    item["preview"]
                ),
            )

        if len(invalid_lines) > 10:
            print(
                "  ...",
                len(invalid_lines) - 10,
                "additional malformed line(s) omitted",
            )

        print(
            "RUN_MODE='update' will rerun any jobs "
            "that exist only in malformed checkpoint records."
        )

    return latest


def append_checkpoint(
    path: Path,
    record: dict[str, Any],
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Serialize completely before touching the checkpoint file. If
    # serialization fails, the existing JSONL remains unchanged.
    payload = (
        json.dumps(
            record,
            ensure_ascii=False,
            default=str,
        )
        + "\n"
    ).encode(
        "utf-8"
    )

    # An interrupted previous append may have left a truncated final
    # record without a newline. Isolate it before appending the next
    # valid JSON object so one bad record cannot swallow a later one.
    if (
        path.exists()
        and path.stat().st_size > 0
    ):
        with path.open(
            "rb"
        ) as handle:
            handle.seek(
                -1,
                os.SEEK_END,
            )
            last_byte = handle.read(
                1
            )

        if last_byte != b"\n":
            with path.open(
                "ab"
            ) as handle:
                handle.write(
                    b"\n"
                )
                handle.flush()
                os.fsync(
                    handle.fileno()
                )

    # Append one complete UTF-8 JSONL record and force it to disk.
    with path.open(
        "ab"
    ) as handle:
        handle.write(
            payload
        )
        handle.flush()
        os.fsync(
            handle.fileno()
        )


async def call_vllm(
    client: httpx.AsyncClient,
    prompt: str,
) -> dict[str, str]:

    endpoint = (
        f"{VLLM_BASE_URL.rstrip('/')}"
        "/chat/completions"
    )

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    SYSTEM_MESSAGE
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {
            "json": (
                RESPONSE_JSON_SCHEMA
            ),
        },
    }

    headers = {
        "Authorization": (
            f"Bearer {VLLM_API_KEY}"
        )
    }

    response = await client.post(
        endpoint,
        json=payload,
        headers=headers,
    )

    response.raise_for_status()

    body = response.json()

    choices = (
        body.get(
            "choices"
        )
        or []
    )

    if not choices:
        raise ValueError(
            "vLLM response "
            "contains no choices"
        )

    choice = choices[0]

    content = (
        choice.get(
            "message",
            {},
        )
        .get(
            "content"
        )
    )

    if not content:
        raise ValueError(
            "vLLM response contains "
            "no message content"
        )

    return {
        "content": str(
            content
        ),
        "finish_reason": str(
            choice.get(
                "finish_reason"
            )
            or ""
        ),
    }


async def classify_one_job(
    job: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:

    common = {
        "job_id": (
            job[
                "job_id"
            ]
        ),
        "document_id": (
            job[
                "document_id"
            ]
        ),
        "source_case_number": (
            job[
                "source_case_number"
            ]
        ),
        "chunk_index": (
            job[
                "chunk_index"
            ]
        ),
        "chunk_count": (
            job[
                "chunk_count"
            ]
        ),
        "input_strategy": (
            job[
                "input_strategy"
            ]
        ),
        "n_supplied_candidates": (
            job[
                "n_supplied_candidates"
            ]
        ),
        "evidence_chars": (
            job[
                "evidence_chars"
            ]
        ),
        "metadata": (
            job[
                "metadata"
            ]
        ),
        "supplied_candidates": (
            job[
                "supplied_candidates"
            ]
        ),
        "model": MODEL,
    }

    prompt = build_prompt(
        job
    )

    last_error = ""

    for attempt in range(
        MAX_RETRIES + 1
    ):
        try:
            async with semaphore:
                result = await call_vllm(
                    client,
                    prompt,
                )

            if (
                result[
                    "finish_reason"
                ]
                == "length"
            ):
                raise RuntimeError(
                    "Model response was "
                    "truncated by max_tokens."
                )

            parsed = (
                extract_json_object(
                    result[
                        "content"
                    ]
                )
            )

            validated = (
                validate_model_response(
                    parsed,
                    job,
                )
            )

            return {
                **common,
                "llm_status": "ok",
                "llm_error": "",
                "classified_at": (
                    utc_now_iso()
                ),
                "finish_reason": (
                    result[
                        "finish_reason"
                    ]
                ),
                "raw_response": (
                    result[
                        "content"
                    ]
                ),
                "parsed_response": (
                    validated
                ),
            }

        except Exception as exc:
            last_error = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            if attempt < MAX_RETRIES:
                await asyncio.sleep(
                    2 ** attempt
                )

    return {
        **common,
        "llm_status": "error",
        "llm_error": (
            last_error
        ),
        "classified_at": (
            utc_now_iso()
        ),
        "finish_reason": "",
        "raw_response": "",
        "parsed_response": None,
    }


async def run_document_jobs(
    jobs: list[
        dict[str, Any]
    ],
) -> list[
    dict[str, Any]
]:

    if RUN_MODE == "redo_all":
        if (
            CHECKPOINT_JSONL_PATH
            .exists()
        ):
            CHECKPOINT_JSONL_PATH.unlink()

    latest = load_latest_checkpoint(
        CHECKPOINT_JSONL_PATH
    )

    if RUN_MODE == "rebuild_excel":
        return list(
            latest.values()
        )

    pending = []

    for job in jobs:
        existing = latest.get(
            job[
                "job_id"
            ]
        )

        if (
            RUN_MODE == "update"
            and existing is not None
            and existing.get(
                "llm_status"
            )
            == "ok"
        ):
            continue

        pending.append(
            job
        )

    print(
        "Successful jobs already checkpointed:",
        sum(
            1
            for record
            in latest.values()
            if record.get(
                "llm_status"
            )
            == "ok"
        ),
    )

    print(
        "Jobs to run:",
        len(
            pending
        ),
    )

    if not pending:
        return list(
            latest.values()
        )

    timeout = httpx.Timeout(
        REQUEST_TIMEOUT_SECONDS
    )

    limits = httpx.Limits(
        max_connections=(
            max(
                CONCURRENT_REQUESTS
                * 2,
                8,
            )
        ),
        max_keepalive_connections=(
            max(
                CONCURRENT_REQUESTS,
                4,
            )
        ),
    )

    semaphore = asyncio.Semaphore(
        CONCURRENT_REQUESTS
    )

    async with httpx.AsyncClient(
        timeout=timeout,
        limits=limits,
    ) as client:

        tasks = [
            asyncio.create_task(
                classify_one_job(
                    job,
                    client,
                    semaphore,
                )
            )
            for job in pending
        ]

        for completed in tqdm(
            asyncio.as_completed(
                tasks
            ),
            total=len(tasks),
            desc="Gemma Court-case classification",
        ):
            record = await completed

            append_checkpoint(
                CHECKPOINT_JSONL_PATH,
                record,
            )

            latest[
                record[
                    "job_id"
                ]
            ] = record

    return list(
        latest.values()
    )


### Optional structured-output smoke test

This sends one synthetic Court-reference job to the local vLLM endpoint and verifies the response parser/schema path. It does not write to the checkpoint.


In [9]:
async def test_structured_output_support():
    test_job = {
        "job_id": "structured-output-test",
        "document_id": "structured-output-test",
        "source_case_number": "T-1/77",
        "chunk_index": 1,
        "chunk_count": 1,
        "input_strategy": "test",
        "metadata": {
            "source_case_number": "T-1/77",
        },
        "supplied_candidates": [
            {
                "provided_candidate": "21/77",
                "regex_family": "legacy_unprefixed",
                "hit_types": "court_legacy_unprefixed",
                "n_regex_hits": 1,
                "matched_texts": "21/77",
                "context_trigger": "case_and_court",
                "regex_confidence_precheck": "medium",
                "regex_self_reference_precheck": False,
                "first_hit_start": 100,
                "context": (
                    "The Court referred to Joined Cases "
                    "21 to 23/77 in its judgment."
                ),
                "paragraph_evidence": "",
            }
        ],
        "n_supplied_candidates": 1,
        "evidence_chars": 100,
    }

    timeout = httpx.Timeout(
        REQUEST_TIMEOUT_SECONDS
    )

    async with httpx.AsyncClient(
        timeout=timeout
    ) as client:

        result = await call_vllm(
            client,
            build_prompt(
                test_job
            ),
        )

    parsed = extract_json_object(
        result[
            "content"
        ]
    )

    validated = (
        validate_model_response(
            parsed,
            test_job,
        )
    )

    print(
        "Structured-output smoke test passed."
    )

    display(
        pd.DataFrame(
            validated[
                "candidate_assessments"
            ]
        )
    )

    return validated


# Optional:
# structured_output_test = await test_structured_output_support()


## 8. Run or resume classification

Run this cell once Gemma is available through the local vLLM endpoint.

For an initial test, you can set `MAX_DOCUMENTS = 20` near the top of the notebook and use `RUN_MODE="redo_all"` with a separate checkpoint filename.


In [10]:
classification_records = (
    await run_document_jobs(
        document_jobs
    )
)

status_counts = pd.Series(
    [
        record.get(
            "llm_status",
            "missing",
        )
        for record
        in classification_records
    ]
).value_counts(
    dropna=False
)

print(
    "Checkpoint records:",
    f"{len(classification_records):,}",
)

print(
    status_counts
)


  line 903 - Expecting value: line 1 column 1 (char 0) - preview: '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
RUN_MODE='update' will rerun any jobs that exist only in malformed checkpoint records.
Successful jobs already checkpointed: 902
Jo

Gemma Court-case classification:   0%|          | 0/7476 [00:00<?, ?it/s]

Checkpoint records: 8,378
error    7476
ok        902
Name: count, dtype: int64


## 9. Rebuild structured outputs from the checkpoint

The final Excel workbook contains:

### `validated_cases`

One row per:

**source Court document × reconstructed target Court case**

This is the main downstream citation table.

### `candidate_validation`

One row per supplied regex candidate, including invalid candidates and the reason for rejection.

### `document_summary`

One row per source document with candidate/reference counts.

### `errors`

Jobs that did not produce a successful validated structured response.


In [11]:
# ---------------------------------------------------------------------
# Latest record per job
# ---------------------------------------------------------------------

latest_by_job = load_latest_checkpoint(
    CHECKPOINT_JSONL_PATH
)

if not latest_by_job and classification_records:
    latest_by_job = {
        record["job_id"]: record
        for record
        in classification_records
        if clean_scalar(
            record.get(
                "job_id"
            )
        )
    }


# ---------------------------------------------------------------------
# Input lookup for audit metadata
# ---------------------------------------------------------------------

candidate_lookup = {}

for _, row in candidates.iterrows():
    key = (
        clean_document_id(
            row[
                "document_id"
            ]
        ),
        clean_scalar(
            row[
                "normalized_candidate"
            ]
        ),
    )

    candidate_lookup[
        key
    ] = row.to_dict()


# ---------------------------------------------------------------------
# Flatten model output
# ---------------------------------------------------------------------

candidate_rows = []
case_rows = []
error_rows = []

for job_id, record in latest_by_job.items():

    status = record.get(
        "llm_status"
    )

    parsed = record.get(
        "parsed_response"
    )

    if (
        status != "ok"
        or not isinstance(
            parsed,
            dict,
        )
    ):
        error_rows.append({
            "job_id": job_id,
            "document_id": (
                record.get(
                    "document_id"
                )
            ),
            "source_case_number": (
                record.get(
                    "source_case_number"
                )
            ),
            "chunk_index": (
                record.get(
                    "chunk_index"
                )
            ),
            "chunk_count": (
                record.get(
                    "chunk_count"
                )
            ),
            "llm_status": status,
            "llm_error": (
                record.get(
                    "llm_error",
                    "",
                )
            ),
            "model": (
                record.get(
                    "model"
                )
            ),
            "classified_at": (
                record.get(
                    "classified_at"
                )
            ),
        })

        continue

    document_id = clean_document_id(
        record.get(
            "document_id"
        )
    )

    source_case_number = (
        clean_scalar(
            record.get(
                "source_case_number"
            )
        )
    )

    metadata = (
        record.get(
            "metadata"
        )
        or {}
    )

    assessments = (
        parsed.get(
            "candidate_assessments"
        )
        or []
    )

    for assessment in assessments:

        provided_candidate = (
            clean_scalar(
                assessment.get(
                    "provided_candidate"
                )
            )
        )

        input_row = candidate_lookup.get(
            (
                document_id,
                provided_candidate,
            ),
            {},
        )

        reconstructed_cases = [
            case.get(
                "case_number"
            )
            for case
            in (
                assessment.get(
                    "court_cases"
                )
                or []
            )
            if case.get(
                "case_number"
            )
        ]

        candidate_rows.append({
            "job_id": job_id,
            "document_id": document_id,
            "source_case_number": (
                source_case_number
            ),
            "chunk_index": (
                record.get(
                    "chunk_index"
                )
            ),
            "chunk_count": (
                record.get(
                    "chunk_count"
                )
            ),
            "provided_candidate": (
                provided_candidate
            ),
            "regex_family": (
                input_row.get(
                    "regex_families",
                    "",
                )
            ),
            "regex_hit_types": (
                input_row.get(
                    "hit_types",
                    "",
                )
            ),
            "n_regex_hits": safe_int(
                input_row.get(
                    "n_hits"
                ),
                0,
            ),
            "regex_context_trigger": (
                input_row.get(
                    "context_triggers",
                    "",
                )
            ),
            "regex_confidence_precheck": (
                input_row.get(
                    "confidence_precheck",
                    "",
                )
            ),
            "regex_self_reference_precheck": (
                input_row.get(
                    "is_exact_self_reference",
                    False,
                )
            ),
            "candidate_is_valid": (
                assessment.get(
                    "candidate_is_valid"
                )
            ),
            "invalid_reason": (
                assessment.get(
                    "invalid_reason"
                )
            ),
            "reference_form": (
                assessment.get(
                    "reference_form"
                )
            ),
            "confidence": (
                assessment.get(
                    "confidence"
                )
            ),
            "reconstructed_case_count": (
                len(
                    reconstructed_cases
                )
            ),
            "reconstructed_cases": (
                json.dumps(
                    reconstructed_cases,
                    ensure_ascii=False,
                )
            ),
            "best_evidence": (
                assessment.get(
                    "best_evidence",
                    "",
                )
            ),
            "explanation": (
                assessment.get(
                    "explanation",
                    "",
                )
            ),
            "context_agg": (
                input_row.get(
                    "context_agg",
                    "",
                )
            ),
            "paragraph_text_agg": (
                input_row.get(
                    "paragraph_text_agg",
                    "",
                )
            ),
            "affId_source": (
                metadata.get(
                    "affId_source",
                    "",
                )
            ),
            "publishedId_source": (
                metadata.get(
                    "publishedId_source",
                    "",
                )
            ),
            "procedureId_source": (
                metadata.get(
                    "procedureId_source",
                    "",
                )
            ),
            "model": (
                record.get(
                    "model"
                )
            ),
            "classified_at": (
                record.get(
                    "classified_at"
                )
            ),
        })

        for case in (
            assessment.get(
                "court_cases"
            )
            or []
        ):
            canonical = (
                normalize_court_case(
                    case.get(
                        "case_number"
                    )
                )
            )

            if canonical is None:
                continue

            case_rows.append({
                "job_id": job_id,
                "document_id": (
                    document_id
                ),
                "source_case_number": (
                    source_case_number
                ),
                "provided_candidate": (
                    provided_candidate
                ),
                "target_court_case": (
                    canonical
                ),
                "target_prefix": (
                    case.get(
                        "prefix",
                        "",
                    )
                ),
                "target_case_number_component": (
                    case.get(
                        "case_number_component",
                        "",
                    )
                ),
                "target_year_component": (
                    case.get(
                        "year_component",
                        "",
                    )
                ),
                "reference_form": (
                    assessment.get(
                        "reference_form"
                    )
                ),
                "reason_added": (
                    case.get(
                        "reason_added"
                    )
                ),
                "is_inferred_from_coordination": (
                    case.get(
                        "is_inferred_from_coordination"
                    )
                ),
                "inherited_elements": (
                    json.dumps(
                        case.get(
                            "inherited_elements"
                        )
                        or [],
                        ensure_ascii=False,
                    )
                ),
                "is_self_reference": (
                    case.get(
                        "is_self_reference"
                    )
                ),
                "confidence": (
                    case.get(
                        "confidence"
                    )
                ),
                "surface_text": (
                    case.get(
                        "surface_text",
                        "",
                    )
                ),
                "best_evidence": (
                    case.get(
                        "best_evidence",
                        "",
                    )
                ),
                "explanation": (
                    case.get(
                        "explanation",
                        "",
                    )
                ),
                "regex_family": (
                    input_row.get(
                        "regex_families",
                        "",
                    )
                ),
                "n_regex_hits": (
                    safe_int(
                        input_row.get(
                            "n_hits"
                        ),
                        0,
                    )
                ),
                "affId_source": (
                    metadata.get(
                        "affId_source",
                        "",
                    )
                ),
                "publishedId_source": (
                    metadata.get(
                        "publishedId_source",
                        "",
                    )
                ),
                "procedureId_source": (
                    metadata.get(
                        "procedureId_source",
                        "",
                    )
                ),
                "model": (
                    record.get(
                        "model"
                    )
                ),
                "classified_at": (
                    record.get(
                        "classified_at"
                    )
                ),
            })


candidate_validation_df = (
    pd.DataFrame(
        candidate_rows
    )
)

raw_case_df = pd.DataFrame(
    case_rows
)

error_df = pd.DataFrame(
    error_rows
)


# ---------------------------------------------------------------------
# Pool to one row per document x target Court case
# ---------------------------------------------------------------------

def ordered_unique(
    values,
) -> list[str]:
    output = []

    for value in values:
        text = clean_scalar(
            value
        )

        if (
            text
            and text not in output
        ):
            output.append(text)

    return output


def highest_confidence(
    values,
) -> str:

    priority = {
        "high": 3,
        "medium": 2,
        "low": 1,
    }

    cleaned = ordered_unique(
        values
    )

    return max(
        cleaned,
        key=lambda value: priority.get(
            value,
            0,
        ),
        default="low",
    )


if raw_case_df.empty:
    validated_cases_df = (
        pd.DataFrame()
    )

else:
    pooled_rows = []

    for (
        document_id,
        target_court_case,
    ), group in raw_case_df.groupby(
        [
            "document_id",
            "target_court_case",
        ],
        sort=True,
        dropna=False,
    ):

        first = group.iloc[0]

        inherited = []

        for raw in group[
            "inherited_elements"
        ]:
            try:
                values = (
                    json.loads(raw)
                    if isinstance(
                        raw,
                        str,
                    )
                    else raw
                )
            except Exception:
                values = [
                    raw
                ]

            for value in (
                values
                or []
            ):
                value = clean_scalar(
                    value
                )

                if (
                    value
                    and value
                    not in inherited
                ):
                    inherited.append(
                        value
                    )

        pooled_rows.append({
            "document_id": (
                document_id
            ),
            "source_case_number": (
                first.get(
                    "source_case_number",
                    "",
                )
            ),
            "target_court_case": (
                target_court_case
            ),
            "target_prefix": (
                first.get(
                    "target_prefix",
                    "",
                )
            ),
            "target_case_number_component": (
                first.get(
                    "target_case_number_component",
                    "",
                )
            ),
            "target_year_component": (
                first.get(
                    "target_year_component",
                    "",
                )
            ),
            "is_self_reference": bool(
                group[
                    "is_self_reference"
                ]
                .map(
                    parse_bool
                )
                .any()
            ),
            "is_inferred_from_coordination": bool(
                group[
                    "is_inferred_from_coordination"
                ]
                .map(
                    parse_bool
                )
                .any()
            ),
            "provided_candidates": (
                json.dumps(
                    ordered_unique(
                        group[
                            "provided_candidate"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "reference_forms": (
                json.dumps(
                    ordered_unique(
                        group[
                            "reference_form"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "reasons_added": (
                json.dumps(
                    ordered_unique(
                        group[
                            "reason_added"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "regex_families": (
                json.dumps(
                    ordered_unique(
                        group[
                            "regex_family"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "inherited_elements": (
                json.dumps(
                    inherited,
                    ensure_ascii=False,
                )
            ),
            "max_confidence": (
                highest_confidence(
                    group[
                        "confidence"
                    ]
                )
            ),
            "surface_forms": (
                json.dumps(
                    ordered_unique(
                        group[
                            "surface_text"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "best_evidence": (
                first_nonempty(
                    group[
                        "best_evidence"
                    ]
                )
            ),
            "explanations": (
                json.dumps(
                    ordered_unique(
                        group[
                            "explanation"
                        ]
                    ),
                    ensure_ascii=False,
                )
            ),
            "affId_source": (
                first.get(
                    "affId_source",
                    "",
                )
            ),
            "publishedId_source": (
                first.get(
                    "publishedId_source",
                    "",
                )
            ),
            "procedureId_source": (
                first.get(
                    "procedureId_source",
                    "",
                )
            ),
        })

    validated_cases_df = (
        pd.DataFrame(
            pooled_rows
        )
        .sort_values(
            [
                "document_id",
                "target_court_case",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ---------------------------------------------------------------------
# Document summary
# ---------------------------------------------------------------------

base_documents = (
    candidates[
        [
            "document_id",
            "source_case_number",
            "affId_source",
            "publishedId_source",
            "procedureId_source",
        ]
    ]
    .drop_duplicates(
        subset=[
            "document_id"
        ]
    )
    .copy()
)


if candidate_validation_df.empty:
    candidate_summary = pd.DataFrame(
        columns=[
            "document_id",
            "n_regex_candidates",
            "n_valid_regex_candidates",
            "n_invalid_regex_candidates",
        ]
    )

else:
    candidate_summary = (
        candidate_validation_df
        .groupby(
            "document_id",
            as_index=False,
        )
        .agg(
            n_regex_candidates=(
                "provided_candidate",
                "nunique",
            ),
            n_valid_regex_candidates=(
                "candidate_is_valid",
                lambda s: int(
                    s.map(
                        parse_bool
                    ).sum()
                ),
            ),
        )
    )

    candidate_summary[
        "n_invalid_regex_candidates"
    ] = (
        candidate_summary[
            "n_regex_candidates"
        ]
        - candidate_summary[
            "n_valid_regex_candidates"
        ]
    )


if validated_cases_df.empty:
    case_summary = pd.DataFrame(
        columns=[
            "document_id",
            "n_validated_court_cases",
            "n_self_references",
            "n_other_court_cases",
            "n_coordination_reconstructed_cases",
        ]
    )

else:
    case_summary = (
        validated_cases_df
        .groupby(
            "document_id",
            as_index=False,
        )
        .agg(
            n_validated_court_cases=(
                "target_court_case",
                "nunique",
            ),
            n_self_references=(
                "is_self_reference",
                lambda s: int(
                    s.map(
                        parse_bool
                    ).sum()
                ),
            ),
            n_coordination_reconstructed_cases=(
                "is_inferred_from_coordination",
                lambda s: int(
                    s.map(
                        parse_bool
                    ).sum()
                ),
            ),
        )
    )

    case_summary[
        "n_other_court_cases"
    ] = (
        case_summary[
            "n_validated_court_cases"
        ]
        - case_summary[
            "n_self_references"
        ]
    )


document_summary_df = (
    base_documents
    .merge(
        candidate_summary,
        on="document_id",
        how="left",
    )
    .merge(
        case_summary,
        on="document_id",
        how="left",
    )
)


for column in [
    "n_regex_candidates",
    "n_valid_regex_candidates",
    "n_invalid_regex_candidates",
    "n_validated_court_cases",
    "n_self_references",
    "n_other_court_cases",
    "n_coordination_reconstructed_cases",
]:
    if column not in document_summary_df.columns:
        document_summary_df[
            column
        ] = 0

    document_summary_df[
        column
    ] = (
        document_summary_df[
            column
        ]
        .fillna(0)
        .astype(int)
    )


document_summary_df[
    "has_validated_court_reference"
] = (
    document_summary_df[
        "n_validated_court_cases"
    ]
    > 0
)

document_summary_df[
    "has_other_court_reference"
] = (
    document_summary_df[
        "n_other_court_cases"
    ]
    > 0
)


# ---------------------------------------------------------------------
# Write workbook
# ---------------------------------------------------------------------

with pd.ExcelWriter(
    OUTPUT_XLSX_PATH,
    engine="openpyxl",
) as writer:

    validated_cases_df.to_excel(
        writer,
        sheet_name="validated_cases",
        index=False,
    )

    candidate_validation_df.to_excel(
        writer,
        sheet_name="candidate_validation",
        index=False,
    )

    document_summary_df.to_excel(
        writer,
        sheet_name="document_summary",
        index=False,
    )

    error_df.to_excel(
        writer,
        sheet_name="errors",
        index=False,
    )

    for sheet_name in [
        "validated_cases",
        "candidate_validation",
        "document_summary",
        "errors",
    ]:
        worksheet = (
            writer.book[
                sheet_name
            ]
        )

        worksheet.freeze_panes = (
            "A2"
        )

        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )


print(
    "Wrote:",
    OUTPUT_XLSX_PATH,
)

print(
    "Validated source×target Court cases:",
    f"{len(validated_cases_df):,}",
)

print(
    "Candidate validation rows:",
    f"{len(candidate_validation_df):,}",
)

print(
    "Errors:",
    f"{len(error_df):,}",
)


  line 903 - Expecting value: line 1 column 1 (char 0) - preview: '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
RUN_MODE='update' will rerun any jobs that exist only in malformed checkpoint records.
Wrote: /home/edik/projects/eccjeu/output/cou

## 10. Quality-control summaries

The most important QA questions are:

1. What share of regex candidates survived?
2. What kinds of false positives were removed?
3. How many references were reconstructed through coordination/ranges?
4. How many source documents have at least one **other** Court case after removing self-references?
5. Are historical prefixless candidates being accepted at sensible rates?


In [12]:
if not candidate_validation_df.empty:

    print(
        "Valid versus invalid regex candidates:"
    )

    display(
        candidate_validation_df[
            "candidate_is_valid"
        ]
        .map(
            parse_bool
        )
        .value_counts(
            dropna=False
        )
        .to_frame(
            "n_candidates"
        )
    )

    print(
        "\nInvalid reasons:"
    )

    display(
        candidate_validation_df[
            ~candidate_validation_df[
                "candidate_is_valid"
            ].map(
                parse_bool
            )
        ][
            "invalid_reason"
        ]
        .value_counts(
            dropna=False
        )
        .to_frame(
            "n_candidates"
        )
    )

    print(
        "\nReference forms:"
    )

    display(
        candidate_validation_df[
            "reference_form"
        ]
        .value_counts(
            dropna=False
        )
        .to_frame(
            "n_candidates"
        )
    )


if not validated_cases_df.empty:

    print(
        "\nValidated Court references:"
    )

    print(
        "Total source×target references:",
        f"{len(validated_cases_df):,}",
    )

    print(
        "Self references:",
        f"{validated_cases_df['is_self_reference'].map(parse_bool).sum():,}",
    )

    print(
        "Other Court references:",
        f"{(~validated_cases_df['is_self_reference'].map(parse_bool)).sum():,}",
    )

    print(
        "Coordination/range reconstructed references:",
        f"{validated_cases_df['is_inferred_from_coordination'].map(parse_bool).sum():,}",
    )

    print(
        "\nConfidence:"
    )

    display(
        validated_cases_df[
            "max_confidence"
        ]
        .value_counts(
            dropna=False
        )
        .to_frame(
            "n_references"
        )
    )

    print(
        "\nSample coordinated reconstructions:"
    )

    display(
        validated_cases_df[
            validated_cases_df[
                "is_inferred_from_coordination"
            ].map(
                parse_bool
            )
        ][
            [
                "document_id",
                "source_case_number",
                "target_court_case",
                "provided_candidates",
                "reference_forms",
                "inherited_elements",
                "max_confidence",
                "best_evidence",
            ]
        ].head(30)
    )


if not document_summary_df.empty:

    print(
        "\nDocuments with at least one validated Court case:"
    )

    print(
        f"{document_summary_df['has_validated_court_reference'].sum():,}"
        f" / {len(document_summary_df):,}"
    )

    print(
        "Documents with at least one OTHER Court case:"
    )

    print(
        f"{document_summary_df['has_other_court_reference'].sum():,}"
        f" / {len(document_summary_df):,}"
    )


if not error_df.empty:

    print(
        "\nLLM / validation errors:"
    )

    display(
        error_df.head(30)
    )


Valid versus invalid regex candidates:


,n_candidates
candidate_is_valid,
True,8044
False,579



Invalid reasons:


,n_candidates
invalid_reason,
regulation_or_directive,387
administrative_or_file_number,148
official_journal_reference,13
date_or_year,13
malformed_unresolvable,8
commission_decision,7
not_court_case,3



Reference forms:


...

,n_references
max_confidence,
high,7790
medium,603



Sample coordinated reconstructions:


,document_id,source_case_number,target_court_case,provided_candidates,reference_forms,inherited_elements,max_confidence,best_evidence
160,1211708,T-210/08,T-82/07,"[""T-81/07""]","[""explicit_prefixed""]","[""T"", ""07""]",medium,Joined Cases T-81/07 to T-83/07
249,1211714,T-185/06,C-206/02,"[""C-205/02""]","[""explicit_prefixed""]","[""C"", ""02""]",medium,C‐205/02 P to C‐208/02 P
250,1211714,T-185/06,C-207/02,"[""C-205/02""]","[""explicit_prefixed""]","[""C"", ""02""]",medium,C‐205/02 P to C‐208/02 P
279,1211716,T-194/06,T-260/02,"[""T-259/02""]","[""explicit_prefixed""]","[""T"", ""02""]",medium,Joined Cases T‐259/02 to T‐264/02
280,1211716,T-194/06,T-261/02,"[""T-259/02""]","[""explicit_prefixed""]","[""T"", ""02""]",medium,Joined Cases T‐259/02 to T‐264/02
281,1211716,T-194/06,T-262/02,"[""T-259/02""]","[""explicit_prefixed""]","[""T"", ""02""]",medium,Joined Cases T‐259/02 to T‐264/02
282,1211716,T-194/06,T-263/02,"[""T-259/02""]","[""explicit_prefixed""]","[""T"", ""02""]",medium,Joined Cases T‐259/02 to T‐264/02
290,1211717,T-196/06,C-206/02,"[""C-208/02""]","[""explicit_prefixed""]","[""C"", ""02""]",medium,C‐205/02 P to C‐208/02 P
291,1211717,T-196/06,C-207/02,"[""C-208/02""]","[""explicit_prefixed""]","[""C"", ""02""]",medium,C‐205/02 P to C‐208/02 P
301,1211717,T-196/06,T-213/98,"[""T-212/98""]","[""explicit_prefixed""]","[""T"", ""98""]",medium,T‐212/98 to T‐214/98



Documents with at least one validated Court case:
875 / 6,321
Documents with at least one OTHER Court case:
658 / 6,321

LLM / validation errors:


,job_id,document_id,source_case_number,chunk_index,chunk_count,llm_status,llm_error,model,classified_at
0,1211704::chunk_001,1211704,T-199/08,1,2,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:37.990145+00:00
1,1055241,1055241,C-112/09 P,1,1,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:37.990172+00:00
2,1211709::chunk_001,1211709,T-192/06,1,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:37.991244+00:00
3,1211709::chunk_002,1211709,T-192/06,2,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.022992+00:00
4,1211711::chunk_002,1211711,T-204/08,2,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.023022+00:00
5,1211705::chunk_002,1211705,T-186/06,2,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.023042+00:00
6,1211718::chunk_002,1211718,T-195/06,2,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.023060+00:00
7,1211724::chunk_001,1211724,T-235/07,1,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.024986+00:00
8,1211727::chunk_001,1211727,T-240/07,1,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.025009+00:00
9,1211738::chunk_001,1211738,T-191/06,1,3,error,ConnectError: All connection attempts failed,google/gemma-4-31B-it,2026-08-31T07:57:38.051283+00:00


## Recommended downstream use

The main downstream table is the `validated_cases` sheet in:

`court_infocuria_court_case_classification_v1.xlsx`

It gives one row per source document × validated/reconstructed Court case.

For a citation network, you will normally exclude:

`is_self_reference == True`

The `candidate_validation` sheet should be retained as the audit trail explaining which regex candidates Gemma rejected and why.

A later stage can map `target_court_case` deterministically against the full InfoCuria case metadata universe and then classify the relationship between the source case and cited Court case if needed.
